# Tabular ML Template Walkthrough

This notebook demonstrates the reusable utilities in `src/` against the bundled SQLite dataset. The flow is intentionally close to what most tabular data science projects need: load data, validate it, inspect it, split features and target, preprocess mixed columns, train baseline models, compare metrics, and persist a model.

In [9]:
from IPython.display import display
from pathlib import Path
from tempfile import TemporaryDirectory

import pandas as pd

from src import (
    CSVDataLoader,
    DataSchema,
    ParquetDataLoader,
    SQLiteDataLoader,
    RANDOM_STATE,
    FeatureColumns,
    build_preprocessor,
    classification_metrics,
    compare_models,
    dataset_summary,
    load_model,
    save_model,
    set_default_feature_columns,
    train_baseline_models,
    train_test_split_dataframe,
    validate_dataframe,
)

pd.set_option("display.max_columns", 50)

## 1. Load Data

`SQLiteDataLoader` resolves paths relative to the project root, lists available tables, and loads a table into a pandas dataframe.

In [10]:
loader = SQLiteDataLoader("data/online_shopping.db")

loader.list_tables()

['online_shopping']

In [14]:
TABLE_NAME = "online_shopping"
TARGET_COLUMN = "PurchaseCompleted"

dataframe = loader.load(TABLE_NAME)

print(f"Shape: {dataframe.shape[0]:,} rows x {dataframe.shape[1]:,} columns")
display(dataframe.head())

Shape: 12,330 rows x 9 columns


,CustomerType,SpecialDayProximity,ExitRate,PageValue,TrafficSource,GeographicRegion,BounceRate,ProductPageTime,PurchaseCompleted
0,Returning_Visitor,0.0,0.20,0.0,1.0,1,0.20,0.000000,0
1,Returning_Visitor,0.0,0.10,0.0,2.0,1,0.00,64.000000,0
2,Returning_Visitor,NaN,0.20,0.0,3.0,-9,0.20,0.000000,0
3,Returning_Visitor,0.0,0.14,0.0,4.0,2,0.05,2.666667,0
4,Returning_Visitor,0.0,NaN,NaN,4.0,1,0.02,627.500000,0


## 3. Validate Expected Data Shape

`DataSchema` is deliberately lightweight. Use it to catch missing columns, dtype changes, null-rate issues, uniqueness violations, and numeric range problems before modeling.

In [15]:
schema = DataSchema(
    required_columns=[
        "CustomerType",
        "SpecialDayProximity",
        "ExitRate",
        "PageValue",
        "TrafficSource",
        "GeographicRegion",
        "BounceRate",
        "ProductPageTime",
        TARGET_COLUMN,
    ],
    dtypes={
        "GeographicRegion": "int64",
        TARGET_COLUMN: "int64",
    },
    null_limits={column: 0.10 for column in dataframe.columns},
    ranges={
        "BounceRate": (0.0, 1.0),
        "ExitRate": (0.0, 1.0),
        "SpecialDayProximity": (0.0, 1.0),
        TARGET_COLUMN: (0, 1),
    },
)

validation_result = validate_dataframe(dataframe, schema, raise_on_error=False)

print("Valid:", validation_result.is_valid)
validation_result.errors


Valid: False


["Column 'BounceRate' contains values below 0.0."]

## 4. Profile the Dataset

`dataset_summary` returns a dictionary with project-friendly profiling outputs. The numeric and categorical sections are pandas dataframes, so they can be displayed or exported directly.

In [16]:
summary = dataset_summary(dataframe)

overview = pd.DataFrame(
    {
        "dtype": summary["dtypes"],
        "missing": summary["missing_values"],
        "missing_fraction": summary["missing_fraction"],
        "cardinality": summary["cardinality"],
    }
)

print("Shape:", summary["shape"])
print("Duplicate rows:", summary["duplicate_rows"])
display(overview)

Shape: (12330, 9)
Duplicate rows: 393


,dtype,missing,missing_fraction,cardinality
CustomerType,str,0,0.000000,8
SpecialDayProximity,float64,616,0.049959,6
ExitRate,float64,616,0.049959,4589
PageValue,float64,616,0.049959,2581
TrafficSource,float64,616,0.049959,20
GeographicRegion,int64,0,0.000000,18
BounceRate,float64,0,0.000000,2011
ProductPageTime,float64,616,0.049959,9128
PurchaseCompleted,int64,0,0.000000,2


In [17]:
display(summary["numeric_stats"])
display(summary["categorical_stats"])

,count,mean,std,min,25%,50%,75%,max
SpecialDayProximity,11714.0,0.060833,0.197459,0.0,0.000000,0.000000,0.000000,1.000000
ExitRate,11714.0,0.043058,0.048506,0.0,0.014286,0.025286,0.050000,0.200000
PageValue,11714.0,5.906249,18.682351,0.0,0.000000,0.000000,0.000000,361.763742
TrafficSource,11714.0,4.059501,4.015758,1.0,2.000000,2.000000,4.000000,20.000000
GeographicRegion,12330.0,2.840308,2.757959,-9.0,1.000000,2.000000,4.000000,9.000000
BounceRate,12330.0,0.020024,0.049423,-0.2,0.000000,0.001961,0.015625,0.200000
ProductPageTime,11714.0,1198.698632,1928.871645,-52.5,184.270833,602.958333,1465.955181,63973.522230
PurchaseCompleted,12330.0,0.154745,0.361676,0.0,0.000000,0.000000,0.000000,1.000000


,count,unique,top,freq
CustomerType,12330,8,Returning_Visitor,10022


## 5. Configure Feature Types

For this template, feature roles are explicit. Define them once with `set_default_feature_columns`, then `build_preprocessor` and `train_baseline_models` reuse the same project-level configuration. `GeographicRegion` is intentionally categorical here even though SQLite loads it as an integer.


In [ ]:
features = dataframe.drop(columns=[TARGET_COLUMN])

feature_columns = set_default_feature_columns(
    numeric_columns=[
        "SpecialDayProximity",
        "ExitRate",
        "PageValue",
        "TrafficSource",
        "BounceRate",
        "ProductPageTime",
    ],
    categorical_columns=[
        "CustomerType",
        "GeographicRegion",
    ],
    boolean_columns=[],
)

feature_columns


## 6. Split and Preprocess

Because the feature roles were set once above, `build_preprocessor` does not need to infer dtypes or receive repeated column lists.

In [ ]:
x_train, x_test, y_train, y_test = train_test_split_dataframe(
    dataframe,
    TARGET_COLUMN,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=True,
)

print(x_train.shape, x_test.shape, y_train.shape, y_test.shape)

In [ ]:
preprocessor = build_preprocessor(x_train, scale_numeric=False)
transformed_preview = preprocessor.fit_transform(x_train.head(10))

print("Transformed preview shape:", transformed_preview.shape)
preprocessor

## 7. Train and Compare Baseline Models

`train_baseline_models` also uses the same default feature-column configuration, so `GeographicRegion` remains categorical during model training.

In [ ]:
modeling_data = dataframe.sample(
    n=min(3_000, len(dataframe)),
    random_state=RANDOM_STATE,
)

x_train, x_test, y_train, y_test = train_test_split_dataframe(
    modeling_data,
    TARGET_COLUMN,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=True,
)

models = train_baseline_models(
    x_train,
    y_train,
    task="classification",
    scale_numeric=True,
)

comparison = compare_models(
    models,
    x_test,
    y_test,
    task="classification",
).sort_values("f1", ascending=False)

comparison

In [ ]:
best_model_name = comparison.index[0]
best_model = models[best_model_name]

predictions = best_model.predict(x_test)
probabilities = best_model.predict_proba(x_test)[:, 1] if hasattr(best_model, "predict_proba") else None

print("Best model:", best_model_name)
classification_metrics(y_test, predictions, probabilities)

## 8. Save and Reload a Model

The saved model is a full scikit-learn pipeline, including preprocessing and the estimator. This demo uses a temporary directory to keep the repository clean.

In [ ]:
with TemporaryDirectory() as tmp_dir:
    model_path = save_model(best_model, "best_model.joblib", base_path=tmp_dir)
    reloaded_model = load_model(model_path)

    reloaded_predictions = reloaded_model.predict(x_test.head(5))

pd.DataFrame(
    {
        "actual": y_test.head(5).to_numpy(),
        "prediction": reloaded_predictions,
    }
)